In [2]:
import torch
import torch.nn as nn
import math



class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_seq_len):

        super().__init__()

        pe = torch.zeros(
            max_seq_len,
            d_model
        )

        position = torch.arange(
            0,
            max_seq_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        seq_len = x.size(1)

        # print(f"pos encoding seq len = {seq_len} and {self.pe[:seq_len]}")
        return x + self.pe[:seq_len]
# re wrting code again and again to get get more familarity 

class MultiHeadAttention(nn.Module):
    def __init__(self, max_seq_len, num_heads, d_model):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model//num_heads

        self.W_Q = nn.Linear(d_model, d_model)

        self.W_K = nn.Linear(d_model, d_model)

        self.W_V = nn.Linear(d_model, d_model)

        self.W_O = nn.Linear(d_model, d_model)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len)).bool()

        self.register_buffer(
            "casual_mask",
            mask
        )

    def forward(self, X):
        batch_size, seq_len, _ = X.shape

        Q = self.W_Q(X)
        K = self.W_K(X)
        V = self.W_V(X) # batch_size, seq_len, d_model

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_k
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_k
        )

        V = V.view( # batch_size, seq_len, num_heads(2), d_model/num_heads (2, 3, 2, 4)
            batch_size,
            seq_len,
            self.num_heads,
            self.d_k
        )

        Q = Q.transpose(1, 2) # 2, 2, 3, 4
        K = K.transpose(1, 2) # 2, 2, 3, 4
        V = V.transpose(1, 2) # 2, 2, 3, 4

        scores = Q @ K.transpose(-2, -1)

        scores = scores/math.sqrt(self.d_k)

        mask = self.casual_mask[:seq_len, :seq_len]

        scores = scores.masked_fill(
            ~mask,
            float("-inf")
        )

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = attention_weights @ V

        output = output.transpose(1, 2)

        output = output.contiguous().view(
            batch_size,
            seq_len,
            self.d_model
        )

        output = self.W_O(output)

        return output



class TransformerBlock(nn.Module):
    def __init__(self, d_model, seq_len, num_heads, d_ff):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)

        self.attention = MultiHeadAttention(
            seq_len,
            num_heads,
            d_model
        )

        self.ln2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        # print(x.shape)

        x = x+ self.attention(self.ln1(x))
        x = x+ self.ffn(self.ln2(x))
        
        return x
class Transformer(nn.Module):

    def __init__(self, vocab_size, d_model, num_heads, d_ff, max_seq_len, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            max_seq_len
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                max_seq_len,
                num_heads,
                d_ff
            ) for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(
            d_model
        )

        self.output_projection = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, token_ids):

        x = self.embedding(token_ids)
        x = self.positional_encoding(x)
        
        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.output_projection(x)

        return logits

# testing

vocab = [
'<PAD>',
'The',
'cat',
'dog',
'sat',
'ran',
'on',
'mat',
'home'
]

d_model = 8
seq_len = 9
num_heads = 2
d_ff = 32

torch.manual_seed(42)

model = Transformer(vocab_size = len(vocab), d_model = d_model, d_ff = d_ff, num_heads = num_heads, max_seq_len = seq_len, num_layers=5)

In [3]:
# backword loop
tokens = torch.tensor([
    [1, 2, 4],
    [1, 3, 5]
])

targets = torch.tensor([
    [2, 4, 6],
    [3, 5, 8]
])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

loss_fn = nn.CrossEntropyLoss()

for epoch in range(2000):

    optimizer.zero_grad()

    logits = model(tokens)

    loss = loss_fn(
        logits.reshape(-1, len(vocab)),
        targets.reshape(-1)
    )

    loss.backward()

    # print(model.blocks[0].attention.W_Q.weight.grad)

    optimizer.step()

    if epoch % 10 ==0:
        print(f"epoch = {epoch}, - loss = {loss.item():.4f}") 

epoch = 0, - loss = 1.9426
epoch = 10, - loss = 1.5780
epoch = 20, - loss = 1.3275
epoch = 30, - loss = 1.1821
epoch = 40, - loss = 1.0799
epoch = 50, - loss = 1.0036
epoch = 60, - loss = 0.9421
epoch = 70, - loss = 0.8860
epoch = 80, - loss = 0.8344
epoch = 90, - loss = 0.7869
epoch = 100, - loss = 0.7428
epoch = 110, - loss = 0.7020
epoch = 120, - loss = 0.6645
epoch = 130, - loss = 0.6300
epoch = 140, - loss = 0.5984
epoch = 150, - loss = 0.5695
epoch = 160, - loss = 0.5431
epoch = 170, - loss = 0.5190
epoch = 180, - loss = 0.4970
epoch = 190, - loss = 0.4770
epoch = 200, - loss = 0.4588
epoch = 210, - loss = 0.4422
epoch = 220, - loss = 0.4271
epoch = 230, - loss = 0.4134
epoch = 240, - loss = 0.4009
epoch = 250, - loss = 0.3894
epoch = 260, - loss = 0.3790
epoch = 270, - loss = 0.3694
epoch = 280, - loss = 0.3607
epoch = 290, - loss = 0.3527
epoch = 300, - loss = 0.3453
epoch = 310, - loss = 0.3386
epoch = 320, - loss = 0.3323
epoch = 330, - loss = 0.3266
epoch = 340, - loss = 0.3

In [4]:
tokens = torch.tensor([
    [1, 2, 4],
    [1, 3, 5]
])

targets = torch.tensor([
    [2, 4, 6],
    [3, 5, 8]
])

model.eval()

with torch.no_grad():

    logits = model(tokens)

    predictions = torch.argmax(
        logits,
        dim=-1
    )

print("Predictions:")
print(predictions)

print("Targets:")
print(targets)

Predictions:
tensor([[2, 4, 6],
        [2, 5, 8]])
Targets:
tensor([[2, 4, 6],
        [3, 5, 8]])


In [5]:
def generate(model, tokens, max_new_tokens):

    model.eval()

    for _ in range(max_new_tokens):

        logits = model(tokens)

        next_token_logits = logits[:, -1, :]

        next_token = torch.argmax(
            next_token_logits,
            dim=-1
        )

        tokens = torch.cat(
            [
                tokens,
                next_token.unsqueeze(1)
            ],
            dim=-1
        )
    return tokens

In [6]:
tokens = torch.tensor([
    [1, 2, 4],
    [1, 3, 5]
])

model.eval()

with torch.no_grad():

    logits = model(tokens)

    next_token_logits = logits[:, -1, :]

    probs = torch.softmax(
        next_token_logits,
        dim=-1
    )

print(probs)

for token_id, probability in enumerate(probs[0]):
    print(
        vocab[token_id],
        probability.item()
    )

tensor([[1.3612e-04, 2.2580e-04, 7.1372e-05, 6.0512e-04, 1.9418e-04, 3.7457e-05,
         9.9789e-01, 2.9825e-04, 5.3781e-04],
        [1.4860e-04, 1.8774e-04, 3.0394e-04, 8.9094e-05, 5.6776e-04, 9.3371e-04,
         6.9487e-04, 5.9295e-05, 9.9701e-01]])
<PAD> 0.00013612340262625366
The 0.0002257968735648319
cat 7.137210923247039e-05
dog 0.0006051248055882752
sat 0.00019417832663748413
ran 3.745739741134457e-05
on 0.9978938698768616
mat 0.0002982469159178436
home 0.000537807063665241


In [7]:
tokens = torch.tensor([
    [1, 2]
])

generated = generate(
    model,
    tokens,
    max_new_tokens=3
)

print(generated)

for token_id in generated[0]:
    print(vocab[token_id])

tensor([[1, 2, 4, 6, 6]])
The
cat
sat
on
on


In [8]:
tokens = torch.tensor([
    [1, 3]
])


generated = generate(
    model,
    tokens,
    max_new_tokens=3
)

print(generated)

for token_id in generated[0]:
    print(vocab[token_id])

tensor([[1, 3, 5, 8, 4]])
The
dog
ran
home
sat
